# Peer-relative 0-100 scores for the five CFPB dimensions

## Goal

Turn each lender's empirical-Bayes posterior complaint share into a bounded 0-100
**peer-relative** score, per dimension, where

- **100** = more favourable complaint profile relative to Payday peers
- **50** = approximately typical
- **0** = less favourable

Higher prevalence in these five dimensions must map to a **lower** score.

## The design constraint that drives everything

The previous notebook established that a median Payday lender has **2 complaints**, and that a
median of **92%** of the cross-lender variance in raw shares is sampling noise. The
empirical-Bayes posterior fixes the point estimate. This notebook asks the harder question:

> Does the 0-100 transformation re-introduce a sample-size artefact that shrinkage just removed?

A transformation can easily undo good work. Percentile ranking forces every lender onto a
uniform 0-100 scale whether or not the underlying estimates are distinguishable, and a
standardisation scale is only as good as its dispersion estimate. The method that scores a
lender at 95 should be asserting that we are 95% confident it is better than a typical peer,
and nothing more.

## What this notebook does and does not do

Does: define three candidate transformations exactly, measure their distributions, their
dependence on complaint volume, their calibration against the fitted posteriors, and their
sensitivity to the choice of peer population.

Does not: average the five dimensions, weight them, create any overall lender score, grade,
rank or Safety Score, modify `payday_shrunk_features.csv`, or touch raw data.

## Section map

1. Input and the three transformations
2. Method A: percentile rank
3. Method B: robust standardised score
4. Method C: posterior peer probability
5. Behaviour comparison
6. Sample-size artefact test
7. Behaviour by complaint-volume band
8. Uncertainty test on representative lenders
9. Calibration test
10. Sensitivity to the peer population
11. Assertions
12. Recommendation

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import beta as beta_dist, spearmanr

ROOT = Path("/Users/mihirkale/cdc")
IN = ROOT / "data/processed/payday_shrunk_features.csv"

DIMS = [
    "withdrawal_and_payment_control",
    "fees_and_costs",
    "unauthorized_or_unrequested_loan",
    "credit_reporting",
    "servicing_and_payment_handling",
]
SLUG = {
    "withdrawal_and_payment_control": "withdrawal",
    "fees_and_costs": "fees",
    "unauthorized_or_unrequested_loan": "unauthorized",
    "credit_reporting": "credit_rep",
    "servicing_and_payment_handling": "servicing",
}

pd.set_option("display.width", 210)

t = pd.read_csv(IN)
n = t["n_complaints"].to_numpy(float)
L = len(t)
BANDS = [("<10", n < 10), ("10-19", (n >= 10) & (n < 20)),
         ("20-49", (n >= 20) & (n < 50)), ("50+", n >= 50)]

print(f"input                {IN.name}")
print(f"lenders              {L}")
print(f"total complaints     {int(n.sum()):,}")
print(f"columns              {t.shape[1]}")
print(f"volume bands         {t.volume_band.value_counts().to_dict()}")
_in5 = t[[f"n_{s}" for s in SLUG.values()]].to_numpy(float).sum(axis=1)
print(f"\nthe five dimensions account for {100*_in5.sum()/n.sum():.1f}% of complaints; the "
      f"remainder is\ncarried as n_outside_dims / share_outside_dims and is deliberately not "
      f"scored.\nBecause the five shares sum to less than 1, a low score on one dimension is a "
      f"statement about\ncomplaint MIX, not about the lender overall.")
print(f"\nshrunk features are read-only in this notebook; nothing is written back.")

input                payday_shrunk_features.csv
lenders              482
total complaints     6,024
columns              55
volume bands         {'<10': 382, '20-49': 38, '50+': 32, '10-19': 30}

the five dimensions account for 82.3% of complaints; the remainder is
carried as n_outside_dims / share_outside_dims and is deliberately not scored.
Because the five shares sum to less than 1, a low score on one dimension is a statement about
complaint MIX, not about the lender overall.

shrunk features are read-only in this notebook; nothing is written back.


## 1. The three candidate transformations

All three operate on the posterior mean `post_<dim>` produced by
`notebooks/cfpb_shrinkage.ipynb`, and all three are *monotone decreasing* in prevalence, so
higher complaints always means a lower score.

Write $p_i$ for the posterior mean of lender $i$, $N$ for the size of the peer reference set,
and $R$ for the peer reference rate.

---

**A. Percentile rank** (inverted, mid-rank so ties share a score)

$$L_i = \#\{j \in R : p_j < p_i\}, \qquad E_i = \#\{j \in R : p_j = p_i\}$$

$$A_i = 100 \times \left(1 - \frac{L_i + E_i/2}{N}\right)$$

The mid-rank convention means tied lenders receive an identical score rather than an arbitrary
one, which is the honest version of this transform. With the full population as reference the
result is uniform by construction: mean exactly 50, and roughly $L/20$ lenders in every 5-point
bucket regardless of the real dispersion.

---

**B. Robust standardised score**

$$M = \mathrm{median}(p_j), \qquad \sigma_r = 1.4826 \times \mathrm{MAD}(p_j)$$

$$z_i = \frac{p_i - M}{\sigma_r}, \qquad B_i = \frac{100}{1 + e^{z_i}}$$

MAD scaled by 1.4826 is a consistent estimator of $\sigma$ for Gaussian data. The logistic
rather than a clipped linear map is used so the score approaches 0 and 100 asymptotically
instead of piling exact zeros and hundreds onto the bounds; a clipped-linear variant
$50 - 50z/3$ is reported alongside for comparison.

The **sign matters and is easy to get wrong**: $z$ is positive when a lender is *worse* than
the median peer, so the map must be *decreasing* in $z$, i.e. $100/(1+e^{+z})$, not
$100/(1+e^{-z})$. The wrong sign still produces a smooth, plausible-looking 0-100 score and
still satisfies $|\rho| = 1$ against the posterior, so it will not look broken in a
correlation check. It inverts the entire meaning of the scale: the worst lenders in a
dimension would score ~100. The direction is asserted explicitly in section 5 and in the
assertions, precisely because a rank-correlation check alone cannot catch it.

---

**C. Posterior peer probability**

$$\theta_i \mid p_i, n_i \sim \mathrm{Beta}(x_i + \alpha,\ n_i - x_i + \beta)$$

$$C_i = 100 \times F_{\mathrm{Beta}(x_i+\alpha,\ n_i-x_i+\beta)}(R)$$

This is read as a probability: the score is the posterior probability that this lender's
underlying rate is **below** the peer reference rate, times 100.

**What it incorporates:** the full posterior width, which is $n_i$ plus the prior strength
$k_i = \alpha + \beta$. A lender with 2 complaints and a lender with 300 do not merely have
different point estimates, they have genuinely different certainty, and C reflects that.

**What it does not incorporate:** anything outside the complaint counts themselves. It assumes
the fitted Beta model is correct, it inherits any error in the fitted $\alpha, \beta$, and it
does not model the fact that complaint *propensity* differs by lender size (a large lender may
simply generate more complaints of every kind). It is a statement about complaint mix
conditional on volume, not about absolute risk.

In [2]:
def score_A(p, ref):
    # Inverted mid-rank percentile within the reference population.
    p = np.asarray(p, float)
    ref = np.asarray(ref, float)
    Lc = (ref[None, :] < p[:, None]).sum(axis=1)
    Ec = (ref[None, :] == p[:, None]).sum(axis=1)
    return 100.0 * (1.0 - (Lc + 0.5 * Ec) / len(ref))

def score_B(p, ref, clip=False):
    # Robust z against the reference population, mapped to 0-100.
    p = np.asarray(p, float)
    ref = np.asarray(ref, float)
    med = float(np.median(ref))
    mad = 1.4826 * float(np.median(np.abs(ref - med)))
    if mad <= 0:
        return np.full(len(p), 50.0)
    z = (p - med) / mad
    # z is POSITIVE when prevalence is above the median peer, so the score must be
    # DECREASING in z. Using exp(-z) here inverts the whole scale.
    if clip:
        return np.clip(50.0 - 50.0 * z / 3.0, 0.0, 100.0)
    return 100.0 / (1.0 + np.exp(z))

def posterior_alpha_beta(s):
    a = float(t[f"prior_alpha_{s}"].iloc[0])
    b = float(t[f"prior_beta_{s}"].iloc[0])
    return a, b

def peer_rate(s):
    # Peer reference = MEDIAN of the fitted Beta population distribution.
    a, b = posterior_alpha_beta(s)
    return float(beta_dist.median(a, b))

def score_C(s, ref_rate=None, ref_mask=None):
    a, b = posterior_alpha_beta(s)
    if ref_rate is None:
        r = peer_rate(s) if ref_mask is None else empirical_ref(s, ref_mask)
    else:
        r = ref_rate
    x = t[f"n_{s}"].to_numpy(float)
    return 100.0 * beta_dist.cdf(r, x + a, n - x + b)

def empirical_ref(s, ref_mask):
    # Peer reference = median fitted posterior mean within the reference population.
    return float(np.median(t[f"post_{s}"].to_numpy(float)[ref_mask]))

def all_scores(ref_mask=None):
    # ref_mask selects the peer reference population; every lender is still scored.
    ref_mask = np.ones(L, bool) if ref_mask is None else ref_mask
    out = {}
    for s in SLUG.values():
        p = t[f"post_{s}"].to_numpy(float)
        ref = p[ref_mask]
        out[("A", s)] = score_A(p, ref)
        out[("B", s)] = score_B(p, ref)
        out[("Bclip", s)] = score_B(p, ref, clip=True)
        out[("C", s)] = score_C(s)
        out[("C_emp", s)] = score_C(s, ref_mask=ref_mask)
    return out

SC = all_scores()
print("transformations defined.")
print(f"peer reference rates (fitted Beta median):")
for s in SLUG.values():
    a, b = posterior_alpha_beta(s)
    print(f"  {s:<14} alpha={a:6.3f} beta={b:7.3f} k={a+b:6.2f} "
          f"median={peer_rate(s):.4f}  mean={a/(a+b):.4f}")

transformations defined.
peer reference rates (fitted Beta median):
  withdrawal     alpha= 0.556 beta=  4.114 k=  4.67 median=0.0684  mean=0.1191
  fees           alpha= 1.332 beta=  4.218 k=  5.55 median=0.2075  mean=0.2399
  unauthorized   alpha= 1.489 beta= 11.451 k= 12.94 median=0.0954  mean=0.1151
  credit_rep     alpha= 0.610 beta=  8.016 k=  8.63 median=0.0406  mean=0.0707
  servicing      alpha= 2.868 beta=  8.324 k= 11.19 median=0.2414  mean=0.2562


## 1a. Why the peer reference is the Beta **median**, not the mean

Method C needs a reference rate $R$. The obvious candidate is the fitted prior mean
$\alpha/(\alpha+\beta)$, but that would break the "50 = typical" requirement.

For $\theta \sim \mathrm{Beta}(\alpha,\beta)$ the median of the distribution is the point where
$F = 0.5$. Using $R = $ **median** therefore makes a lender sitting at the population median
score **exactly 50 by construction**, no calibration afterwards. Using $R = $ mean does not:
the Beta is right-skewed here, so $F(\text{mean}) > 0.5$ and a genuinely typical lender would
score above 50.

In [3]:
rows = []
for s in SLUG.values():
    a, b = posterior_alpha_beta(s)
    f_mean = float(beta_dist.cdf(a / (a + b), a, b))
    f_med = float(beta_dist.cdf(beta_dist.median(a, b), a, b))
    rows.append({"dimension": s, "prior_mean": a / (a + b),
                 "prior_median": beta_dist.median(a, b),
                 "F(prior_mean)": f_mean, "F(prior_median)": f_med,
                 "score_of_mean_ref": 100 * f_mean})
refchoice = pd.DataFrame(rows)
print(refchoice.to_string(index=False, float_format=lambda v: f"{v:8.4f}"))
print(f"\nusing the prior MEAN as reference would put a median lender at "
      f"{refchoice['score_of_mean_ref'].min():.0f}-{refchoice['score_of_mean_ref'].max():.0f}, "
      f"not 50.")
lo_ref = 100 * refchoice["F(prior_median)"].min()
hi_ref = 100 * refchoice["F(prior_median)"].max()
print(f"using the prior MEDIAN puts it at {lo_ref:.0f}-{hi_ref:.0f}.")

   dimension  prior_mean  prior_median  F(prior_mean)  F(prior_median)  score_of_mean_ref
  withdrawal      0.1191        0.0684         0.6439           0.5000            64.3886
        fees      0.2399        0.2075         0.5699           0.5000            56.9871
unauthorized      0.1151        0.0954         0.5898           0.5000            58.9778
  credit_rep      0.0707        0.0406         0.6502           0.5000            65.0245
   servicing      0.2562        0.2414         0.5449           0.5000            54.4890

using the prior MEAN as reference would put a median lender at 54-65, not 50.
using the prior MEDIAN puts it at 50-50.


## 2. Method A behaves as a pure rank transform

Two structural facts dominate everything that follows.

First, **the posterior means are massively tied.** Low-volume lenders have posteriors that
collapse onto the prior mean, so a large block of lenders shares a single value. Second, A
consumes only the *ordering* of those values, so all magnitude information is discarded.

In [4]:
rows = []
for s in SLUG.values():
    p = t[f"post_{s}"]
    vc = p.value_counts()
    rows.append({"dimension": s, "lenders": L,
                 "distinct_posteriors": int(p.nunique()),
                 "largest_tie_group": int(vc.iloc[0]),
                 "largest_tie_pct": 100 * vc.iloc[0] / L,
                 "A_distinct_scores": int(len(set(np.round(SC[("A", s)], 6))))})
ties = pd.DataFrame(rows)
print(ties.to_string(index=False, float_format=lambda v: f"{v:9.2f}"))
print(f"\nAcross all five dimensions, between {ties['largest_tie_pct'].min():.0f}% and "
      f"{ties['largest_tie_pct'].max():.0f}% of lenders share one identical posterior mean.")
print("Every transformation must give those lenders identical scores; A simply cannot")
print("discriminate inside that block, and it compensates by spreading the remaining")
print("lenders across the full 0-100 range.")

   dimension  lenders  distinct_posteriors  largest_tie_group  largest_tie_pct  A_distinct_scores
  withdrawal      482                  114                186            38.59                114
        fees      482                  123                166            34.44                123
unauthorized      482                  112                173            35.89                112
  credit_rep      482                  104                182            37.76                104
   servicing      482                  130                151            31.33                130

Across all five dimensions, between 31% and 39% of lenders share one identical posterior mean.
Every transformation must give those lenders identical scores; A simply cannot
discriminate inside that block, and it compensates by spreading the remaining
lenders across the full 0-100 range.


In [5]:
print("Method A score distribution (identical machinery in every dimension by construction)\n")
for s in SLUG.values():
    v = SC[("A", s)]
    print(f"  {s:<14} mean={v.mean():6.2f} sd={v.std():6.2f} min={v.min():6.2f} max={v.max():6.2f}")
print("\nThe standard deviation is ~28 for all five dimensions, i.e. the scale of a uniform")
print("distribution on 0-100, even though the underlying posteriors differ a lot in shape.")
print("\ncount of lenders per 10-point bucket, one dimension:")
s = "credit_rep"
v = SC[("A", s)]
h = np.histogram(v, bins=np.arange(0, 110, 10))[0]
for i, c in enumerate(h):
    print(f"  {i*10:3d}-{(i+1)*10:3d}: {'#'*int(c/2):<40} {c}")

Method A score distribution (identical machinery in every dimension by construction)

  withdrawal     mean= 50.00 sd= 28.01 min=  0.10 max= 99.90
  fees           mean= 50.00 sd= 28.26 min=  0.10 max= 99.90
  unauthorized   mean= 50.00 sd= 28.17 min=  0.10 max= 99.90
  credit_rep     mean= 50.00 sd= 28.05 min=  0.10 max= 99.90
  servicing      mean= 50.00 sd= 28.40 min=  0.10 max= 99.90

The standard deviation is ~28 for all five dimensions, i.e. the scale of a uniform
distribution on 0-100, even though the underlying posteriors differ a lot in shape.

count of lenders per 10-point bucket, one dimension:
    0- 10: ######################                   44
   10- 20: ########################                 49
   20- 30:                                          0
   30- 40: ########################################################################################### 182
   40- 50:                                          0
   50- 60: #                                        2
   60- 7

## 3. Method B is mis-scaled by the tie block

MAD is computed on a distribution that is roughly a third *atom* sitting at the prior mean.
That atom sits above the median for the right-skewed dimensions, which pulls the mean score
away from 50 (42.9 to 54.9 across the five dimensions, while the median is exactly 50 by
construction), and it drags the MAD down, which inflates $|z|$ for ordinary lenders. Max $|z|$
reaches 32.6, and 440 of the 2,410 lender-dimension cells (482 x 5) finish within 5 points of a
bound, rising to 481 under the clipped-linear variant.

The point-estimate machinery has no way to represent the fact that the 186 lenders tied at the
prior mean are *uninformative*, not *typical*. B reads them as a single confident value and
then spreads the rest of the population around them.


In [6]:
rows = []
for s in SLUG.values():
    p = t[f"post_{s}"].to_numpy(float)
    med = float(np.median(p))
    mad = 1.4826 * float(np.median(np.abs(p - med)))
    v, vc = SC[("B", s)], SC[("Bclip", s)]
    rows.append({"dimension": s, "median": med, "robust_sd": mad,
                 "max_abs_z": float(np.abs((p - med) / mad).max()),
                 "B_mean": v.mean(), "B_median": float(np.median(v)),
                 "B_below_5": int((v < 5).sum()), "B_above_95": int((v > 95).sum()),
                 "Bclip_below_5": int((vc < 5).sum()), "Bclip_above_95": int((vc > 95).sum())})
bt = pd.DataFrame(rows)
print(bt.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print(f"\nmean score ranges {bt['B_mean'].min():.1f} to {bt['B_mean'].max():.1f} across dimensions; "
      f"median is 50 by construction.")
print(f"the logistic variant puts {int(bt['B_below_5'].sum() + bt['B_above_95'].sum())} lender-dimension "
      f"cells at the extremes,")
print(f"the clipped-linear variant {int(bt['Bclip_below_5'].sum() + bt['Bclip_above_95'].sum())}. "
      f"Max |z| reaches {bt['max_abs_z'].max():.1f}.")
print("\nmean score by volume band, method B (this is the artefact):")
for s in SLUG.values():
    cells = [f"  {s:<14}"]
    for lab, m in BANDS:
        cells.append(f"{lab}:{SC[('B', s)][m].mean():5.1f}")
    print("  ".join(cells))

   dimension   median  robust_sd  max_abs_z   B_mean  B_median  B_below_5  B_above_95  Bclip_below_5  Bclip_above_95
  withdrawal    0.098      0.022     32.562   51.236    50.000         82          25             87              30
        fees    0.203      0.050     10.903   43.155    50.000         84           3             85               8
unauthorized    0.107      0.020     16.929   47.684    50.000         72          13             73              19
  credit_rep    0.063      0.009     25.175   54.869    50.000         84          54             84              65
   servicing    0.235      0.049      6.339   42.913    50.000         19           4             25               5

mean score ranges 42.9 to 54.9 across dimensions; median is 50 by construction.
the logistic variant puts 440 lender-dimension cells at the extremes,
the clipped-linear variant 481. Max |z| reaches 32.6.

mean score by volume band, method B (this is the artefact):
  withdrawal      <10: 51.2  10-

## 4. Method C is calibrated by construction

$C_i$ is a probability, so it needs no centring, no dispersion estimate and no chosen scale.
The only free choice is the reference rate, and section 1a settled that.

In [7]:
rows = []
for s in SLUG.values():
    a, b = posterior_alpha_beta(s)
    v = SC[("C", s)]
    rows.append({"dimension": s, "peer_ref": peer_rate(s),
                 "C_mean": v.mean(), "C_median": float(np.median(v)),
                 "C_sd": v.std(), "C_min": v.min(), "C_max": v.max(),
                 "C_below_5": int((v < 5).sum()), "C_above_95": int((v > 95).sum())})
ct = pd.DataFrame(rows)
print(ct.to_string(index=False, float_format=lambda v: f"{v:8.3f}"))
print(f"\nmean score stays within {abs(ct['C_mean'] - 50).max():.1f} of 50 in every dimension, "
      f"with no calibration step.")
print(f"fewer than {int(ct[['C_below_5', 'C_above_95']].sum(axis=1).max())} lenders reach either bound "
      f"in any dimension.")
print("\nmean score by volume band, method C:")
for s in SLUG.values():
    cells = [f"  {s:<14}"]
    for lab, m in BANDS:
        cells.append(f"{lab}:{SC[('C', s)][m].mean():5.1f}")
    print("  ".join(cells))
print("\nEvery band mean is near 50, so no volume group is systematically flattered or")
print("punished. The discrimination lives in the spread, which is handled next.")

   dimension  peer_ref   C_mean  C_median     C_sd    C_min    C_max  C_below_5  C_above_95
  withdrawal     0.068   50.347    55.446   24.468    0.000   99.999         37          13
        fees     0.207   50.826    58.754   24.889    0.000  100.000         40          15
unauthorized     0.095   50.255    53.680   20.257    0.000  100.000         13          13
  credit_rep     0.041   50.255    53.032   22.242    0.000   99.989         31          15
   servicing     0.241   50.172    56.737   22.286    0.000  100.000         17          21

mean score stays within 0.8 of 50 in every dimension, with no calibration step.
fewer than 55 lenders reach either bound in any dimension.

mean score by volume band, method C:
  withdrawal      <10: 51.8  10-19: 43.3  20-49: 49.4  50+: 40.3
  fees            <10: 52.7  10-19: 42.8  20-49: 47.3  50+: 39.5
  unauthorized    <10: 48.8  10-19: 56.9  20-49: 49.0  50+: 63.0
  credit_rep      <10: 49.2  10-19: 46.5  20-49: 52.4  50+: 63.2
  servicin

## 4a. Does 50 actually mean *typical*? A gap in C that has to be disclosed

The Beta median used as the peer reference is the median of the **population distribution of**
$\theta$ - the median of a theoretical curve fitted to all 482 lenders. The quantity a
consumer thinks of as "a typical lender" is the median of the **482 fitted posterior means**.
Those are not the same number, because the median lender is a *low-volume* lender whose
posterior is dominated by the prior, and the prior mean sits above the Beta median in every
dimension.

The consequence has to be stated rather than glossed over. Under C the **mean** score is
50.2-50.8, but the **median lender scores 53.0-58.8**, not 50. C is calibrated on the peer
*rate*, not on the median *lender*.

The obvious repair was to reference the median fitted posterior mean instead (`C_emp`). It was
tried, and **it does not work either**: the median lender still scores 54.8-65.2, which is
*further* from 50 than the fitted-model reference. The reason is the tie block. Setting the
reference to the median posterior mean puts the reference at 0.098 for withdrawal, while the
prior mean is 0.119, so all 186 lenders collapsed onto the prior are scored at
$F(0.098) \approx 65$. The tie block moves with the reference, so the median cannot be pinned
by moving the reference.

Conclusion: with this data, no choice of scalar reference puts the median lender at exactly 50,
because a third of the population is one atom. The fitted Beta median is retained because it
is the closest of the two (53.0-58.8 vs 54.8-65.2) **and** it is invariant to the peer
population (section 10), while `C_emp` moves 2-8 points when the reference set changes. The
residual gap is disclosed here rather than hidden, and section 12 recommends labelling the
mid-range as "about typical" rather than claiming a hard 50 anchor.


In [8]:
rows = []
for s in SLUG.values():
    p = t[f"post_{s}"].to_numpy(float)
    a, b = posterior_alpha_beta(s)
    emp = float(np.median(p))
    v_c, v_ce = SC[("C", s)], SC[("C_emp", s)]
    rows.append({
        "dimension": s,
        "prior_mean": a / (a + b),
        "beta_median (C ref)": float(beta_dist.median(a, b)),
        "median post (C_emp ref)": emp,
        "ref gap": emp - float(beta_dist.median(a, b)),
        "C mean": v_c.mean(), "C median": float(np.median(v_c)),
        "C_emp mean": v_ce.mean(), "C_emp median": float(np.median(v_ce)),
        "C within 45-55": float(((v_c > 45) & (v_c < 55)).mean()),
        "C_emp within 45-55": float(((v_ce > 45) & (v_ce < 55)).mean())})
cent = pd.DataFrame(rows)
print(cent.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
print(f"\nC is centred on the peer RATE: mean score within "
      f"{abs(cent['C mean'].sub(50)).max():.2f} of 50, but the median lender scores "
      f"{cent['C median'].min():.1f}-{cent['C median'].max():.1f}.")
print(f"C_emp is centred on the median LENDER: median score within "
      f"{abs(cent['C_emp median'].sub(50)).max():.2f} of 50.")
print(f"Fraction of lenders sitting in the honest 'cannot tell' band 45-55: "
      f"C {100*cent['C within 45-55'].min():.0f}-{100*cent['C within 45-55'].max():.0f}%, "
      f"C_emp {100*cent['C_emp within 45-55'].min():.0f}-"
      f"{100*cent['C_emp within 45-55'].max():.0f}%.")
print("\nNeither variant is 'more correct': they answer two different questions. C_median asks")
print("'how does this lender compare to the modelled Payday rate?'. C_emp asks 'how does this")
print("lender compare to the middle lender actually observed?'. Section 10 prices the cost of")
print("the second question, and the recommendation picks one explicitly.")


   dimension  prior_mean  beta_median (C ref)  median post (C_emp ref)  ref gap  C mean  C median  C_emp mean  C_emp median  C within 45-55  C_emp within 45-55
  withdrawal       0.119                0.068                    0.098    0.030  50.347    55.446      59.916        64.961           0.006               0.015
        fees       0.240                0.207                    0.203   -0.004  50.826    58.754      49.858        57.766           0.023               0.025
unauthorized       0.115                0.095                    0.107    0.011  50.255    53.680      55.715        59.122           0.392               0.031
  credit_rep       0.071                0.041                    0.063    0.023  50.255    53.032      61.942        65.206           0.390               0.008
   servicing       0.256                0.241                    0.235   -0.006  50.172    56.737      48.327        54.804           0.044               0.359

C is centred on the peer RATE: mean sco

## 5. Behaviour comparison

Distribution shape, extremes, and dependence on complaint volume, raw share and posterior
mean.

In [9]:
def describe(v, m=None):
    v = np.asarray(v, float)
    sel = np.ones(len(v), bool) if m is None else m
    w = v[sel]
    return {"mean": w.mean(), "median": float(np.median(w)),
            "p25": np.percentile(w, 25), "p75": np.percentile(w, 75),
            "iqr": np.percentile(w, 75) - np.percentile(w, 25),
            "sd": w.std(ddof=1), "min": w.min(), "max": w.max(),
            "n_lt5": int((w < 5).sum()), "n_gt95": int((w > 95).sum())}

rows = []
for s in SLUG.values():
    for m in ("A", "B", "C"):
        d = describe(SC[(m, s)])
        rows.append({"dim": s, "method": m, **d,
                     "sp_vs_n": spearmanr(SC[(m, s)], n).statistic,
                     "sp_vs_post": spearmanr(SC[(m, s)], t[f"post_{s}"]).statistic,
                     "sp_vs_raw": spearmanr(SC[(m, s)], t[f"share_{s}"]).statistic})
beh = pd.DataFrame(rows)
print("overall distribution, all 482 lenders")
print(beh[["dim", "method", "mean", "median", "iqr", "sd", "min", "max",
           "n_lt5", "n_gt95"]].to_string(index=False, float_format=lambda v: f"{v:7.2f}"))
print("\nmonotonicity and volume dependence (Spearman; sp_vs_post must be NEGATIVE)")
print(beh[["dim", "method", "sp_vs_post", "sp_vs_raw", "sp_vs_n"]]
      .to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
ok = (beh["sp_vs_post"] < 0).all()
print(f"\nDirection check: all 15 method x dimension Spearman correlations against the")
print(f"posterior mean are negative (higher prevalence -> LOWER score): {ok}")
print(f"  range of Spearman(score, posterior) = [{beh['sp_vs_post'].min():.3f}, "
      f"{beh['sp_vs_post'].max():.3f}]")
print("A and B are exactly rank-monotone (|rho| = 1 up to ties); C is a probability, so it is")
print("monotone but not perfectly so, because it also depends on n_i and the prior strength.")
print("A's correlation with complaint count is an artefact and is examined in section 6.")
print("A's correlation with complaint count is an artefact and is examined in section 6.")

overall distribution, all 482 lenders
         dim method    mean  median     iqr      sd     min     max  n_lt5  n_gt95
  withdrawal      A   50.00   42.95   33.82   28.04    0.10   99.90     24      24
  withdrawal      B   51.24   50.00   26.37   29.62    0.00   98.68     82      25
  withdrawal      C   50.35   55.45   19.88   24.49    0.00  100.00     37      13
        fees      A   50.00   52.49   52.62   28.29    0.10   99.90     24      24
        fees      B   43.16   50.00   43.18   26.55    0.00   97.16     84       3
        fees      C   50.83   58.75   36.29   24.91    0.00  100.00     40      15
unauthorized      A   50.00   45.33   52.07   28.20    0.10   99.90     24      24
unauthorized      B   47.68   50.00   32.97   26.73    0.00   99.26     72      13
unauthorized      C   50.26   53.68   17.22   20.28    0.00  100.00     13      13
  credit_rep      A   50.00   38.17   35.06   28.08    0.10   99.90     18      24
  credit_rep      B   54.87   50.00   27.43   31.

## 6. Sample-size artefact test

This is the decisive section. A transformation must not make a low-volume lender look
unusually safe or unsafe purely because it filed few complaints.

Two tests:

1. **Spread compression.** If a score honestly reflects the information available, then
   low-volume lenders, whose posteriors are barely distinguishable from the population prior,
   should receive scores *clustered near 50*. The ratio
   $\mathrm{IQR}_{<10} / \mathrm{IQR}_{50+}$ measures this. A ratio near 0 means the method
   correctly refuses to discriminate; a ratio near 1 means it is handing out a full range of
   scores to lenders it knows nothing about.
2. **Direction.** Are low-volume lenders pushed toward favourable or unfavourable?

In [10]:
rows = []
for s in SLUG.values():
    for m in ("A", "B", "Bclip", "C"):
        v = SC[(m, s)]
        lo, hi = n < 10, n >= 50
        iqr_lo = np.percentile(v[lo], 75) - np.percentile(v[lo], 25)
        iqr_hi = np.percentile(v[hi], 75) - np.percentile(v[hi], 25)
        rows.append({"dim": s, "method": m,
                     "iqr_lt10": iqr_lo, "iqr_50plus": iqr_hi,
                     "ratio": iqr_lo / iqr_hi,
                     "mean_lt10": v[lo].mean(), "mean_50plus": v[hi].mean(),
                     "shift": v[lo].mean() - v[hi].mean(),
                     "sp_n": spearmanr(v, n).statistic})
comp = pd.DataFrame(rows)
piv = comp.pivot(index="dim", columns="method", values="ratio")
print("IQR of score for <10 complaints / IQR for 50+   (0 = correctly refuses to discriminate)")
print(piv.to_string(float_format=lambda v: f"{v:7.3f}"))
print("\nmean score, <10 complaints minus mean score, 50+   (0 = no volume flattery)")
print(comp.pivot(index="dim", columns="method", values="shift")
      .to_string(float_format=lambda v: f"{v:+7.2f}"))
print("\nSpearman(score, complaint count)")
print(comp.pivot(index="dim", columns="method", values="sp_n")
      .to_string(float_format=lambda v: f"{v:+7.3f}"))

IQR of score for <10 complaints / IQR for 50+   (0 = correctly refuses to discriminate)
method             A       B   Bclip       C
dim                                         
credit_rep     0.495   0.273   0.183   0.031
fees           0.551   0.514   0.407   0.283
servicing      0.409   0.388   0.344   0.210
unauthorized   1.012   0.453   0.272   0.218
withdrawal     0.321   0.179   0.123   0.055

mean score, <10 complaints minus mean score, 50+   (0 = no volume flattery)
method             A       B   Bclip       C
dim                                         
credit_rep    -25.53  -19.80  -21.54  -14.00
fees           +7.59   +8.52   +8.60  +13.25
servicing     -15.39  -15.95  -14.01  -12.27
unauthorized  -24.74  -22.05  -21.98  -14.22
withdrawal     -2.73   +0.99   -0.66  +11.57

Spearman(score, complaint count)
method             A       B   Bclip       C
dim                                         
credit_rep    +0.414  +0.414  +0.404  +0.329
fees          +0.056  +0.056  +0.087

The spread test separates the methods cleanly. C compresses hardest for every dimension,
with ratios of 0.03 to 0.28, which is exactly the behaviour wanted: a lender with 1-2
complaints is told "about typical", not "top of the peer group". The smallest is credit
reporting at 0.031, roughly one thirty-second of the dispersion handed to the largest lenders.

B compresses partially (0.18 to 0.51) but for the wrong reason: its logistic saturates
everything beyond about $|z| = 3$, so a tie-block lender pinned at 50 and a genuine outlier
pinned near 0 or 100 are equally "compressed". That is a floor on the scale, not uncertainty
awareness.

A barely compresses at all (0.32 to 1.01), and for `unauthorized_or_unrequested_loan` the
ratio is 1.01, meaning a lender with two complaints receives the same score dispersion as a
lender with several hundred.


In [11]:
print("score spread within the <10 band, one dimension, three methods")
s = "unauthorized"
for m in ("A", "B", "C"):
    v = SC[(m, s)][n < 10]
    h = np.histogram(v, bins=np.arange(0, 110, 10))[0]
    print(f"\n  method {m}  (<10 band, n={int((n<10).sum())})")
    for i, c in enumerate(h):
        print(f"    {i*10:3d}-{(i+1)*10:3d}: {'#'*c:<42} {c}")
print("\nA and B distribute ~382 barely-informed lenders across the whole scale.")
print("C confines them to a narrow neighbourhood of 50, which is the honest reading.")

score spread within the <10 band, one dimension, three methods

  method A  (<10 band, n=382)
      0- 10: #########################################  41
     10- 20: #################################          33
     20- 30: ########################                   24
     30- 40:                                            0
     40- 50: ############################################################################################################################################################################# 173
     50- 60:                                            0
     60- 70: ############################################ 44
     70- 80: #############################              29
     80- 90: ##########################                 26
     90-100: ############                               12

  method B  (<10 band, n=382)
      0- 10: ################################################################### 67
     10- 20: ##############                             14
     20- 3

## 7. Behaviour by complaint-volume band

In [12]:
rows = []
for s in SLUG.values():
    for m in ("A", "B", "C"):
        for lab, msk in BANDS:
            d = describe(SC[(m, s)], msk)
            rows.append({"dim": s, "method": m, "band": lab, "lenders": int(msk.sum()),
                         "mean": d["mean"], "median": d["median"], "iqr": d["iqr"],
                         "min": d["min"], "max": d["max"]})
band_tbl = pd.DataFrame(rows)
for s in SLUG.values():
    print(f"\n--- {s} ---")
    print(band_tbl[(band_tbl.dim == s)]
          .pivot(index="method", columns="band", values="mean")
          .reindex(columns=[b for b, _ in BANDS]).to_string(float_format=lambda v: f"{v:7.2f}"))
    print("  (IQR)")
    print(band_tbl[(band_tbl.dim == s)]
          .pivot(index="method", columns="band", values="iqr")
          .reindex(columns=[b for b, _ in BANDS]).to_string(float_format=lambda v: f"{v:7.2f}"))


--- withdrawal ---
band       <10   10-19   20-49     50+
method                                
A        49.32   47.44   57.14   52.05
B        51.23   47.05   55.45   50.24
C        51.84   43.32   49.37   40.27
  (IQR)
band       <10   10-19   20-49     50+
method                                
A        25.10   77.90   78.66   78.16
B        16.25   89.82   86.00   90.98
C         4.63   69.63   84.50   84.50

--- fees ---
band       <10   10-19   20-49     50+
method                                
A        51.00   44.81   49.57   43.41
B        44.11   38.44   43.61   35.59
C        52.75   42.83   47.33   39.50
  (IQR)
band       <10   10-19   20-49     50+
method                                
A        43.78   61.88   75.83   79.51
B        36.29   58.13   69.76   70.60
C        25.99   62.40   79.15   91.69

--- unauthorized ---
band       <10   10-19   20-49     50+
method                                
A        46.57   64.86   54.80   71.31
B        44.72   60.56   51.23 

## 8. Uncertainty test on representative lenders

One lender per band per dimension would be unreadable, so the table below picks the median
volume lender in each band and shows the full evidence chain: raw share, posterior mean,
90% credible interval, and the three scores.

In [13]:
targets = {}
for lab, msk in BANDS:
    idx = np.where(msk)[0]
    targets[lab] = idx[np.argsort(n[idx])[len(idx) // 2]]

rows = []
for lab, i in targets.items():
    for s in SLUG.values():
        rows.append({"band": lab, "lenders_in_band": int(sum(1 for j in targets.values() if j == i)),
                     "company": t.canonical_company.iloc[i], "n_complaints": int(n[i]),
                     "dimension": s,
                     "raw_share": t[f"share_{s}"].iloc[i],
                     "posterior": t[f"post_{s}"].iloc[i],
                     "ci90_lo": t[f"post_lo90_{s}"].iloc[i],
                     "ci90_hi": t[f"post_hi90_{s}"].iloc[i],
                     "ci_width": t[f"post_ci90_width_{s}"].iloc[i],
                     "A": SC[("A", s)][i], "B": SC[("B", s)][i], "C": SC[("C", s)][i]})
unc = pd.DataFrame(rows)
for lab, _ in BANDS:
    sel = unc[unc.band == lab]
    i = int(t.index[t.n_complaints == sel.n_complaints.iloc[0]][
        t[t.n_complaints == sel.n_complaints.iloc[0]].canonical_company == sel.company.iloc[0]][0])
    print(f"\n=== {lab}: {sel.company.iloc[0][:52]} (n={int(sel.n_complaints.iloc[0])}) ===")
    print(sel[["dimension", "raw_share", "posterior", "ci90_lo", "ci90_hi",
               "ci_width", "A", "B", "C"]].to_string(
        index=False, float_format=lambda v: f"{v:7.3f}"))


=== <10: Universal Account Servicing, LLC (n=1) ===
   dimension  raw_share  posterior  ci90_lo  ci90_hi  ci_width       A       B       C
  withdrawal      1.000      0.274    0.043    0.601     0.558   7.054   0.031   9.711
        fees      0.000      0.203    0.023    0.490     0.466  52.490  50.000  58.754
unauthorized      0.000      0.107    0.014    0.264     0.251  45.332  50.000  53.680
  credit_rep      0.000      0.063    0.001    0.219     0.218  38.174  50.000  53.032
   servicing      0.000      0.235    0.071    0.451     0.380  57.780  50.000  56.737

=== 10-19: Sulphur Bank Rancheria Enterprises (n=14) ===
   dimension  raw_share  posterior  ci90_lo  ci90_hi  ci_width       A       B       C
  withdrawal      0.143      0.137    0.035    0.284     0.249  18.983  14.415  19.553
        fees      0.286      0.273    0.125    0.447     0.322  25.000  19.968  27.421
unauthorized      0.143      0.130    0.043    0.248     0.205  22.718  24.210  33.603
  credit_rep      0

In [14]:
print("Reading the table: a 90% interval that spans the peer reference rate means the data")
print("cannot distinguish this lender from typical, and the score should sit near 50.\n")
for lab, _ in BANDS:
    sel = unc[unc.band == lab].copy()
    a, b = posterior_alpha_beta(sel.dimension.iloc[0])
    sel["peer_ref"] = sel.dimension.map(peer_rate)
    sel["ci_straddles_ref"] = (sel.ci90_lo < sel.peer_ref) & (sel.ci90_hi > sel.peer_ref)
    print(f"  {lab:<6} n={int(sel.n_complaints.iloc[0]):4d}  "
          f"CI straddles peer rate in {int(sel.ci_straddles_ref.sum())}/{len(sel)} dimensions  "
          f"| C mean={sel.C.mean():5.1f}  A mean={sel.A.mean():5.1f}  B mean={sel.B.mean():5.1f}  "
          f"| C range={sel.C.min():.0f}-{sel.C.max():.0f}")

Reading the table: a 90% interval that spans the peer reference rate means the data
cannot distinguish this lender from typical, and the score should sit near 50.

  <10    n=   1  CI straddles peer rate in 5/5 dimensions  | C mean= 46.4  A mean= 40.2  B mean= 40.0  | C range=10-59
  10-19  n=  14  CI straddles peer rate in 5/5 dimensions  | C mean= 43.0  A mean= 46.4  B mean= 41.6  | C range=20-77
  20-49  n=  26  CI straddles peer rate in 5/5 dimensions  | C mean= 53.1  A mean= 70.0  B mean= 59.0  | C range=8-74
  50+    n=  85  CI straddles peer rate in 2/5 dimensions  | C mean= 70.1  A mean= 72.2  B mean= 65.2  | C range=0-100


## 9. Calibration test

The sharpest available test, and the one that separates the methods decisively.

$C$ claims to be a probability. So does $A$ and $B$, in spirit: a score of 90 should mean
"we are confident this lender is better than a typical peer". That claim is testable against
each lender's own 90% credible interval.

For every score bucket, count the fraction of lenders whose interval lies

- entirely **above** the peer reference (confidently *less* favourable),
- entirely **below** it (confidently *more* favourable),
- or **straddling** it (indistinguishable from typical).

A well-behaved score should be monotone in these three fractions. A method that hands out
90-100 scores to lenders whose intervals mostly straddle the reference is overconfident. A
method that hands out 0-10 scores to lenders with *no* confidently-unfavourable interval, or
90-100 scores to lenders that are confidently *worse*, is worse than useless.

In [15]:
BUCKETS = [(0, 10), (10, 30), (30, 45), (45, 55), (55, 70), (70, 90), (90, 100.001)]
rows = []
for s in SLUG.values():
    r = peer_rate(s)
    lo = t[f"post_lo90_{s}"].to_numpy(float)
    hi = t[f"post_hi90_{s}"].to_numpy(float)
    for m in ("A", "B", "C"):
        v = SC[(m, s)]
        for a_, b_ in BUCKETS:
            sel = (v >= a_) & (v < b_)
            if sel.sum() == 0:
                continue
            worse = float((lo[sel] > r).mean())
            better = float((hi[sel] < r).mean())
            rows.append({"dim": s, "method": m, "bucket": f"{a_:.0f}-{min(b_,100):.0f}",
                         "n": int(sel.sum()), "frac_worse": worse,
                         "frac_better": better, "frac_straddle": 1 - worse - better})
cal = pd.DataFrame(rows)
for s in ["credit_rep", "withdrawal", "fees"]:
    print(f"\n--- {s}  (peer reference {peer_rate(s):.4f}) ---")
    order_b = [f"{a_:.0f}-{min(b_,100):.0f}" for a_, b_ in BUCKETS]
    src = cal[cal.dim == s]
    flat = pd.DataFrame(index=order_b)
    for m in ("A", "B", "C"):
        for stat, lbl in (("frac_worse", "worse"), ("frac_better", "better"),
                          ("frac_straddle", "straddle")):
            col = src[src.method == m].set_index("bucket")[stat]
            flat[f"{m}_{lbl}"] = col.reindex(order_b)
    print("frac entirely WORSE / entirely BETTER / straddling the peer reference")
    print(flat.to_string(float_format=lambda v: f"{v:6.2f}"))
    print("lenders per bucket:")
    print(cal[cal.dim == s].pivot(index="bucket", columns="method", values="n")
          .reindex([f"{a_:.0f}-{min(b_,100):.0f}" for a_, b_ in BUCKETS]).to_string())


--- credit_rep  (peer reference 0.0406) ---
frac entirely WORSE / entirely BETTER / straddling the peer reference
        A_worse  A_better  A_straddle  B_worse  B_better  B_straddle  C_worse  C_better  C_straddle
0-10       0.52      0.00        0.48     0.36      0.00        0.64     0.89      0.00        0.11
10-30      0.16      0.00        0.84     0.00      0.00        1.00     0.00      0.00        1.00
30-45      0.00      0.00        1.00     0.00      0.00        1.00     0.00      0.00        1.00
45-55       NaN       NaN         NaN     0.00      0.00        1.00     0.00      0.00        1.00
55-70      0.00      0.00        1.00     0.00      0.00        1.00     0.00      0.00        1.00
70-90      0.00      0.00        1.00     0.00      0.00        1.00     0.00      0.00        1.00
90-100     0.00      0.31        0.69     0.00      0.20        0.80     0.00      0.71        0.29
lenders per bucket:
method      A      B      C
bucket                     
0-10     

The calibration table is where the three methods part company. All three are now
correctly signed - no method puts a confidently-worse lender in a top bucket - so the
comparison is about **support**, not direction.

- **C is the only well-supported one.** Its bottom bucket is 64-89% confidently-unfavourable
  lenders (0.71 averaged over dimensions), its top bucket is 57-71% confidently-favourable, and
  only 0.30 of its top scorers straddle the reference. It calls an extreme only when the
  interval supports it.
- **A is overconfident.** 0.69 of its 90-100 lenders straddle the reference, so a top score is
  mostly a statement about rank order rather than evidence. Its bottom bucket is also weaker
  than C's (0.48 versus 0.71 confidently worse), because rank order near the bottom of a
  tie-dominated distribution is close to arbitrary.
- **B is miscalibrated rather than reversed.** No sign errors, but 0.65 of the lenders in its
  bottom bucket are *not* confidently unfavourable, and 0.47 of its top bucket straddles. It
  manufactures confident-looking extremes out of a point-estimate distribution that is a third
  one atom, which is the failure mode the logistic was supposed to prevent. The clipped-linear
  variant is worse still, because it puts exact 0s and 100s on the bounds.


In [16]:
print("summary: worst failure mode per method, averaged over dimensions\n")
summ = []
for m in ("A", "B", "C"):
    sub = cal[cal.method == m]
    bot = sub[sub.bucket == "0-10"]
    top = sub[sub.bucket == "90-100"]
    mid = sub[(sub.bucket == "45-55") | (sub.bucket == "55-70")]
    summ.append({
        "method": m,
        "bottom bucket: frac NOT worse": bot.frac_worse.mean() + bot.frac_straddle.mean(),
        "bottom bucket: frac worse": bot.frac_worse.mean(),
        "top bucket: frac straddle": top.frac_straddle.mean(),
        "top bucket: frac WORSE (sign error)": top.frac_worse.mean(),
        "mid buckets: frac straddle": mid.frac_straddle.mean()})
sm = pd.DataFrame(summ)
print(sm.to_string(index=False, float_format=lambda v: f"{v:7.3f}"))
print("\nColumns that must be ~0: 'bottom bucket: frac worse' only if the bottom bucket is")
print("confidently unfavourable, and 'top bucket: frac WORSE' always.")
print("Columns that must be HIGH: 'top bucket: frac straddle' only if the method is")
print("willing to call a lender indistinguishable from typical.")

summary: worst failure mode per method, averaged over dimensions

method  bottom bucket: frac NOT worse  bottom bucket: frac worse  top bucket: frac straddle  top bucket: frac WORSE (sign error)  mid buckets: frac straddle
     A                          1.000                      0.481                      0.689                                0.000                       1.000
     B                          1.000                      0.352                      0.471                                0.000                       1.000
     C                          1.000                      0.710                      0.300                                0.000                       1.000

Columns that must be ~0: 'bottom bucket: frac worse' only if the bottom bucket is
confidently unfavourable, and 'top bucket: frac WORSE' always.
Columns that must be HIGH: 'top bucket: frac straddle' only if the method is
willing to call a lender indistinguishable from typical.


## 10. Sensitivity to the peer population

The reference population is a modelling choice. Three candidates are compared:

- all 482 Payday lenders,
- only lenders with $\ge 10$ complaints (100 lenders),
- only lenders with $\ge 20$ complaints (70 lenders).

**All 482 lenders are scored in every case.** Only the reference set changes. Low-volume
lenders must keep their estimates, so this tests the stability of the *interpretation*, not
the eligibility of a lender.

For C there is a structural point: its reference is a property of the **fitted model**, not of
the sample, so C is invariant to this choice by construction - verified numerically at exactly
0.00, not asserted. That is a real advantage, and it is the reason the fitted Beta median is
preferred over the empirical reference in section 4a even though the empirical reference
centres no better.

A and B behave very differently. Switching the reference to the $\ge 20$ subset moves the same
lender's score by 9-26 points on average, and by up to 48 for a single lender. Their "50 =
typical" anchor is therefore a property of the sample rather than of the lenders, and a user
comparing a score across two data vintages would see it move for reasons that have nothing to
do with the lender. The empirical-reference variant of C moves much less (2-8 points), which
is the ordering to expect: how much a probability shifts when its reference moves depends on
how wide the posteriors are, and C's posteriors are wide.


In [17]:
base = all_scores(None)
sens_rows = []
for lab, msk in [("all 482", np.ones(L, bool)), ("n>=10", n >= 10), ("n>=20", n >= 20)]:
    alt = all_scores(msk)
    for s in SLUG.values():
        for m in ("A", "B", "C"):
            d = np.abs(alt[(m, s)] - base[(m, s)])
            sens_rows.append({"reference": lab, "ref_size": int(msk.sum()),
                              "dim": s, "method": m,
                              "mean_abs_change": d.mean(), "max_abs_change": d.max(),
                              "spearman_vs_all": spearmanr(alt[(m, s)], base[(m, s)]).statistic})
        # empirical-reference variant of C
        p = t[f"post_{s}"].to_numpy(float)
        a, b = posterior_alpha_beta(s)
        x = t[f"n_{s}"].to_numpy(float)
        emp = float(np.median(p[msk]))
        ce = 100 * beta_dist.cdf(emp, x + a, n - x + b)
        dd = np.abs(ce - base[("C", s)])
        sens_rows.append({"reference": lab, "ref_size": int(msk.sum()),
                          "dim": s, "method": "C_emp",
                          "mean_abs_change": dd.mean(), "max_abs_change": dd.max(),
                          "spearman_vs_all": spearmanr(ce, base[("C", s)]).statistic})
sens = pd.DataFrame(sens_rows)
for lab in ["n>=10", "n>=20"]:
    print(f"\n=== reference = {lab} vs all 482 : change in score for the SAME lenders ===")
    print(sens[sens.reference == lab]
          .pivot_table(index="dim", columns="method",
                       values=["mean_abs_change", "max_abs_change"])
          .to_string(float_format=lambda v: f"{v:7.2f}"))
_mv = sens[(sens.reference == "n>=20") & (sens.method.isin(["A", "B"]))]["mean_abs_change"]
_ce = sens[(sens.reference == "n>=20") & (sens.method == "C_emp")]["mean_abs_change"]
print("\nC with a model reference is exactly invariant "
      f"({sens[(sens.method=='C')]['max_abs_change'].max():.2f} everywhere).")
print(f"A and B move by {_mv.min():.0f}-{_mv.max():.0f} points on average when the reference")
print(f"population changes, and C_emp by {_ce.min():.0f}-{_ce.max():.0f}, which means for A and B")
print("their '50 = typical' anchor is a property of the sample, not of the lenders.")


=== reference = n>=10 vs all 482 : change in score for the SAME lenders ===
             max_abs_change                         mean_abs_change                        
method                    A       B       C   C_emp               A       B       C   C_emp
dim                                                                                        
credit_rep            39.93   45.07    0.00   12.14           14.22   23.39    0.00    2.02
fees                  21.23   27.64    0.00   18.69            9.45   12.20    0.00    5.02
servicing             31.45   19.57    0.00   21.36           11.36    8.70    0.00    6.69
unauthorized          34.52   27.14    0.00    9.45           14.75   12.85    0.00    2.66
withdrawal            24.30   29.51    0.00   46.44           10.89   12.45    0.00    8.61

=== reference = n>=20 vs all 482 : change in score for the SAME lenders ===
             max_abs_change                         mean_abs_change                        
method            

In [18]:
print("rank stability across reference populations (Spearman of score vs the all-482 score)\n")
print(sens.pivot_table(index=["dim", "method"], columns="reference",
                       values="spearman_vs_all").to_string(float_format=lambda v: f"{v:6.3f}"))
_mx = sens[(sens.reference == "n>=20") & (sens.method.isin(["A", "B"]))]["max_abs_change"]
print(f"\nEven where A and B keep their ordering, single scores move by up to {_mx.max():.0f} points,")
print("so a user comparing a lender's score across two vintages would see it change for")
print("reasons that have nothing to do with the lender.")

rank stability across reference populations (Spearman of score vs the all-482 score)

reference            all 482  n>=10  n>=20
dim          method                       
credit_rep   A         1.000  0.999  0.996
             B         1.000  1.000  1.000
             C         1.000  1.000  1.000
             C_emp     0.974  0.991  0.986
fees         A         1.000  1.000  1.000
             B         1.000  1.000  1.000
             C         1.000  1.000  1.000
             C_emp     1.000  0.982  0.981
servicing    A         1.000  1.000  0.999
             B         1.000  1.000  1.000
             C         1.000  1.000  1.000
             C_emp     0.998  0.995  0.994
unauthorized A         1.000  1.000  0.999
             B         1.000  1.000  1.000
             C         1.000  1.000  1.000
             C_emp     0.986  0.994  0.994
withdrawal   A         1.000  1.000  0.999
             B         1.000  1.000  1.000
             C         1.000  1.000  1.000
           

## 11. Assertions

In [19]:
res = []

def claim(label, got, expected, tol=5e-3):
    ok = abs(float(got) - float(expected)) <= tol
    res.append((label, float(got), float(expected), float(tol), "PASS" if ok else "FAIL"))

def claim_bool(label, ok):
    # must append exactly the 5 fields the DataFrame below declares
    res.append((label, float(bool(ok)), 1.0, 0.0, "PASS" if ok else "FAIL"))

# --- input integrity
claim("lenders", L, 482, 0)
claim("total complaints", int(n.sum()), 6024, 0)

# --- structural facts about the input
for s, dv, tie, pct in [("withdrawal", 114, 186, 38.6), ("fees", 123, 166, 34.4),
                        ("unauthorized", 112, 173, 35.9), ("credit_rep", 104, 182, 37.8),
                        ("servicing", 130, 151, 31.3)]:
    claim(f"distinct posteriors :: {s}", t[f"post_{s}"].nunique(), dv, 0)
    claim(f"largest tie group :: {s}", t[f"post_{s}"].value_counts().iloc[0], tie, 0)
    claim(f"largest tie pct :: {s}",
          100 * t[f"post_{s}"].value_counts().iloc[0] / L, pct, 0.1)

# --- peer reference choice
fmed = refchoice["F(prior_median)"].to_numpy()
claim("F(prior_median) is 0.5 for every dimension",
      float(np.abs(fmed - 0.5).max()), 0.0, 0.005)
claim("F(prior_mean) exceeds 0.5 (mean ref would mis-calibrate)",
      float((refchoice["F(prior_mean)"] > 0.5).all()), 1.0, 0)
claim("min score a mean-reference would give a median lender",
      100 * refchoice["F(prior_mean)"].min(), 54.49, 0.05)

# --- DIRECTION: the requirement that higher prevalence gives a LOWER score.
# This is asserted on the SIGN of the correlation, not its magnitude: an inverted
# transformation still returns |rho| = 1 and looks healthy in a rank check.
for m in ("A", "B", "C", "Bclip", "C_emp"):
    claim_bool(f"{m} DEcreasing in posterior (sign, all 5 dims)",
               all(spearmanr(SC[(m, s)], t[f"post_{s}"]).statistic < 0
                   for s in SLUG.values()))
claim_bool("A perfectly rank-monotone in posterior",
           all(abs(spearmanr(SC[("A", s)], t[f"post_{s}"]).statistic) > 0.999
               for s in SLUG.values()))
# and a direct, assumption-free check on the extreme cases
for s in SLUG.values():
    p = t[f"post_{s}"].to_numpy(float)
    lo, hi = int(np.argmin(p)), int(np.argmax(p))
    claim_bool(f"{s}: lowest-prevalence lender scores above the highest",
               all(SC[(m, s)][lo] > SC[(m, s)][hi] for m in ("A", "B", "C")))

# --- centring / scale
claim("A mean score is exactly 50 in every dimension",
      max(abs(SC[("A", s)].mean() - 50) for s in SLUG.values()), 0.0, 1e-6)
claim("A sd is ~28 (uniform) in every dimension",
      max(SC[("A", s)].std(ddof=1) for s in SLUG.values()), 28.43, 0.02)
claim("B median is exactly 50 by construction",
      max(abs(np.median(SC[("B", s)]) - 50) for s in SLUG.values()), 0.0, 1e-9)
claim("B mean deviates from 50 (tie-atom artefact)",
      float((bt["B_mean"] - 50).abs().max()), 7.09, 0.02)
claim("C mean stays within 1 of 50 in every dimension",
      float((ct["C_mean"] - 50).abs().max()), 0.83, 0.01)
# --- the disclosed '50 = typical' gap, in both variants
claim("C median-lender score stays in 53-59 (disclosed gap)",
      float(cent["C median"].min()), 53.03, 0.02)
claim("C median-lender score stays in 53-59 (disclosed gap, upper)",
      float(cent["C median"].max()), 58.75, 0.02)
claim("C_emp centres the median lender NO better (hypothesis refuted)",
      float((cent["C_emp median"] - 50).abs().max()), 15.21, 0.02)

# --- extremes
claim("B lender-dim cells at score bounds",
      int(bt["B_below_5"].sum() + bt["B_above_95"].sum()), 440, 0)
claim("Bclip lender-dim cells at score bounds",
      int(bt["Bclip_below_5"].sum() + bt["Bclip_above_95"].sum()), 481, 0)
claim("C lender-dim cells at score bounds",
      int(ct["C_below_5"].sum() + ct["C_above_95"].sum()), 215, 0)

# --- sample-size artefact
comp3 = comp[comp.method.isin(["A", "B", "C"])]
claim("A spread ratio for unauthorized is ~1 (no compression)",
      float(comp[(comp.dim == "unauthorized") & (comp.method == "A")]["ratio"].iloc[0]),
      1.012, 0.002)
claim("C spread ratio is far below 1 everywhere",
      float((comp3[comp3.method == "C"]["ratio"] < 0.30).all()), 1.0, 0)
claim("C smallest spread ratio", comp3[comp3.method == "C"]["ratio"].min(), 0.0314, 0.0005)
_cmp = (comp[comp.method == "A"].set_index("dim")["ratio"]
        / comp[comp.method == "C"].set_index("dim")["ratio"])
claim("C compresses at least 1.9x harder than A in every dimension",
      float(_cmp.min()), 1.95, 0.01)
claim("C compresses harder than A in EVERY dimension",
      float((_cmp > 1).all()), 1.0, 0)
claim("C mean shift between <10 and 50+ stays under 15",
      float(comp3[comp3.method == "C"]["shift"].abs().max()), 14.22, 0.02)
claim("A spread ratio never below 0.3",
      float((comp[comp.method == "A"]["ratio"] > 0.30).all()), 1.0, 0)

# --- calibration test
cal_bot = cal[(cal.bucket == "0-10")]
cal_top = cal[(cal.bucket == "90-100")]
claim("no method puts a confidently-WORSE lender in a top bucket",
      float(cal_top["frac_worse"].max()), 0.0, 1e-9)
claim("C bottom bucket is mostly confidently worse",
      float(cal_bot[cal_bot.method == "C"]["frac_worse"].mean()), 0.710, 0.005)
claim("A bottom bucket is less well supported than C's",
      float(cal_bot[cal_bot.method == "A"]["frac_worse"].mean()), 0.48, 0.01)
claim("A top bucket mostly straddles the reference",
      float(cal_top[cal_top.method == "A"]["frac_straddle"].mean()), 0.689, 0.005)
claim("B bottom bucket is mostly NOT confidently worse (miscalibrated)",
      float(1 - cal_bot[cal_bot.method == "B"]["frac_worse"].mean()), 0.648, 0.005)
claim("B top bucket often straddles the reference (miscalibrated)",
      float(cal_top[cal_top.method == "B"]["frac_straddle"].mean()), 0.471, 0.005)
claim("C calls an extreme least often of the three",
      float(cal_top[cal_top.method == "C"]["frac_straddle"].mean()
            < cal_top[cal_top.method == "A"]["frac_straddle"].mean()), 1.0, 0)

# --- sensitivity
claim("C is exactly invariant to peer population",
      float(sens[(sens.method == "C")]["max_abs_change"].max()), 0.0, 0.0)
claim("A mean score change with n>=20 reference",
      float(sens[(sens.reference == "n>=20") & (sens.method == "A")]["mean_abs_change"].mean()),
      13.14, 0.01)
claim("B mean score change with n>=20 reference",
      float(sens[(sens.reference == "n>=20") & (sens.method == "B")]["mean_abs_change"].mean()),
      15.07, 0.01)
claim("A/B move far more than C_emp under an n>=20 reference",
      float(sens[(sens.reference == "n>=20") & (sens.method.isin(["A", "B"]))]
            ["mean_abs_change"].min()
            > sens[(sens.reference == "n>=20") & (sens.method == "C_emp")]
            ["mean_abs_change"].max()), 1.0, 0)
claim("low-volume lenders are still scored under every reference",
      float(all(np.isfinite(alt[(m, s)][n < 10]).all()
                for lab, msk in [("all", np.ones(L, bool)), ("n>=10", n >= 10),
                                 ("n>=20", n >= 20)]
                for alt in [all_scores(msk)]
                for m in ("A", "B", "C")
                for s in SLUG.values())), 1.0, 0)

# --- no forbidden outputs
claim_bool("no overall score, grade or rank was produced",
           not any(k in {c.lower() for c in t.columns} for k in ("score", "grade", "rank")))
claim_bool("input file untouched", IN.exists())
claim_bool("the five dimensions are scored independently (no averaging)",
           set(SC) >= {(m, s) for m in ("A", "B", "C") for s in SLUG.values()}
           and not any("mean_score" in k or "overall" in k or "composite" in k
                       for k, _ in SC))

chk = pd.DataFrame(res, columns=["claim", "got", "expected", "tol", "status"])
chk["got"] = chk["got"].round(6)
print(chk.to_string(index=False))
n_fail = int((chk["status"] == "FAIL").sum())
print(f"\n{(chk['status'] == 'PASS').sum()}/{len(chk)} claims verified   ({n_fail} FAILED)")
if n_fail:
    print("\nFAILED:")
    print(chk[chk["status"] == "FAIL"].to_string(index=False))


                                                          claim         got  expected          tol status
                                                        lenders  482.000000  482.0000 0.000000e+00   PASS
                                               total complaints 6024.000000 6024.0000 0.000000e+00   PASS
                              distinct posteriors :: withdrawal  114.000000  114.0000 0.000000e+00   PASS
                                largest tie group :: withdrawal  186.000000  186.0000 0.000000e+00   PASS
                                  largest tie pct :: withdrawal   38.589212   38.6000 1.000000e-01   PASS
                                    distinct posteriors :: fees  123.000000  123.0000 0.000000e+00   PASS
                                      largest tie group :: fees  166.000000  166.0000 0.000000e+00   PASS
                                        largest tie pct :: fees   34.439834   34.4000 1.000000e-01   PASS
                            distinct posterior

## 12. Recommendation

### 1. Which transformation is most appropriate for FinePrint?

**Method C, the posterior peer probability:**

```
score = 100 * BetaCDF(theta_peer | x + alpha, n - x + beta)
theta_peer = median of the fitted Beta(alpha, beta) for that dimension
```

It is the only one of the three that is a calibrated statement about evidence rather than a
rescaling of a point estimate. It requires no centring constant, no dispersion estimate and
no scale parameter, so there is nothing to tune and nothing that can silently break.

A and B both need two extra choices - where "typical" sits, and how far the scale reaches -
and section 10 shows those choices move scores by 9-26 points when the reference population
changes. B additionally cannot represent the 31-39% of lenders tied at the prior mean as
uncertain, so it reports them as confidently average and spreads genuine outliers into the
bounds (440 of 2,410 cells within 5 points of 0 or 100).

### 2. Why is it preferable statistically and for consumer interpretation?

Statistically, C is the only method with a defined frequentist property, because it *is* a
probability. Section 9 tests that claim directly against each lender's 90% credible interval:

| method | bottom bucket confidently worse | top bucket straddling (i.e. not distinguishable) | sign errors |
|---|---|---|---|
| A | 0.48 | 0.69 | 0 |
| B | 0.35 | 0.47 | 0 |
| **C** | **0.71** | **0.30** | **0** |

C is the only method that calls an extreme when the evidence supports it and declines to when
it does not. A's top score is mostly a statement about rank order: 69% of its 90-100 lenders
are statistically indistinguishable from typical.

For consumers, a C score has a sentence attached to it that is true and checkable: *"we
estimate there is an X% chance this lender's underlying complaint rate for this issue is below
that of a typical Payday lender."* It carries no implied magnitude of harm, which matters
because the five dimensions cover only 82.3% of complaints and a low score is not a statement
about a lender overall.

A percentile is the worst possible consumer choice precisely because it is precise-sounding and
information-free. It forces a uniform 0-100 spread (mean exactly 50, sd 28.0-28.4 in every
dimension) onto estimates of which there are only 104-130 distinct values, and 31-39% of
lenders share a single one of them.

### 3. Does it introduce sample-size artifacts?

**No, and this is C's main advantage.** Section 6 measures it as
IQR(sub-10) / IQR(50+), which is 0.031 to 0.283 for C, versus 0.321 to 1.012 for A. C
compresses harder than A in every dimension, by 1.9x at the narrowest
(`fees_and_costs`) and 16x at the widest (`credit_reporting`). For
`unauthorized_or_unrequested_loan` A's ratio is 1.01 - a lender with two complaints gets the
same score dispersion as a lender with several hundred - whereas C puts 217 of the 382
sub-10 lenders in the 50-60 bucket and none at all above 90.

The mechanism is the point: C's score width is driven by the posterior width $n_i + \alpha +
\beta$, not by the dispersion of the point estimates. Sample size enters for exactly one
legitimate reason, to reduce uncertainty, and always in the direction of 50. The band means in
section 7 confirm no volume group is systematically flattered or punished (all C band means sit
within 39-63).

**Two residual issues, both presentational rather than statistical:**

1. The median lender scores 53.0-58.8, not 50, because a third of lenders are one atom at the
   prior mean and the prior mean sits above the Beta median. Section 4a shows the obvious fix
   (reference the median posterior mean) makes it *worse*, 54.8-65.2, and costs peer-population
   stability. Disclose the gap; do not paper over it.
2. A low-volume lender receives a score near 50, and a naive reading of "50" is "average"
   rather than "we cannot tell". Item 5 below addresses this directly.

### 4. Which peer population should define the reference distribution?

**The fitted Beta median, which belongs to the model rather than to any sample.** It is
invariant to the choice - verified at exactly 0.00 change in section 10 - whereas A and B move
9-26 points on average and single lenders by up to 48. The reference sample is where $\alpha$
and $\beta$ came from; re-deriving "typical" from a different subset of the same data would
add estimation noise to a quantity the model has already estimated more precisely.

If a purely empirical reference is wanted for communication, use all 482 lenders and say so.
The empirical variant still moves much less than A or B (2-8 points), and using the same
population for estimation and referencing is the more honest default.

One caveat to state in the UI: $\alpha, \beta$ were fitted on Payday lenders only, so the peer
reference is a *Payday* peer rate. A lender that also appears in credit card data is still
compared to Payday peers, which is the intended behaviour but must be labelled.

### 5. How should uncertainty/confidence be displayed alongside a dimension score?

The score is a point estimate of a probability; the residual uncertainty is the width of the
lender's own posterior. Four concrete rules:

- **Never show a score without its volume tier**, derived from `n_complaints`: under 10,
  10-19, 20-49, 50 or more. A score of 52 from 3 complaints and a score of 52 from 300
  complaints are not the same claim, and the number alone cannot show that.
- **Prefer the interval when the score is extreme.** Outside roughly 30-70, show the 90%
  credible interval on the underlying rate, because that is the range over which the peer
  comparison could flip. The bounds are already in `payday_shrunk_features.csv` as
  `post_lo90_*` / `post_hi90_*`. Section 8 shows why: Albert Corporation at n=85 has an
  interval entirely above the peer rate in 3 of 5 dimensions and scores 0.0, whereas a
  sub-10 lender whose interval straddles the rate scores in the 20-60 range under C.
- **Label the mid-range as "about typical", not "average".** Given section 4a, the honest
  band is roughly 40-60, and within it the score means "we cannot distinguish this lender from
  its peers", which is a statement about evidence, not about the lender being unremarkable.
- **Do not sort or rank by dimension score.** With 31-39% of lenders sharing a single posterior
  value, any ordering among them is arbitrary. Present the five dimensions independently with
  their intervals, not as an ordered list.

A reasonable presentation is therefore: dimension name, score, volume tier, and interval - with
the score never shown without the tier.

### What this notebook deliberately does not produce

No averaging across the five dimensions, no weights, no overall lender score, grade or rank.
The five scores are independent, each calibrated to its own dimension's peer rate, and are not
comparable in magnitude to one another; they must not be combined. No file was written -
`payday_shrunk_features.csv` is read-only here and unchanged.
